# Predicting Titled Tuesday game outcomes

**Goal:** predict the outcome of a Chess.com Titled Tuesday blitz game —
**win / draw / loss from White's perspective** — using only information available
**before the first move**.

This notebook is the writeup. It loads the assembled dataset, trains a small ladder
of models, evaluates them on two splits, and gives an honest read on quality and
next steps. All the logic lives in the `stock_fisher` package; here we just call
it and narrate.

- **Data:** five Titled Tuesday events, one per month (Feb–Jun 2026) — the brief
  suggests two; five gives more data and a more robust temporal split. Mined from
  the Chess.com Published-Data API and assembled by `uv run chess-dataset`.
- **Rebuild from scratch:** `uv run chess-dataset` (dataset) then this notebook.

In [1]:
import pandas as pd
from stock_fisher.config import OUTPUT_DIR
from stock_fisher.modeling import (
    load_dataset, feature_columns, pooled_split, temporal_split,
    default_model_specs, fit_evaluate,
)
from stock_fisher.modeling.data import LABELS

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", None)

df = load_dataset(OUTPUT_DIR / "titled_tuesday.parquet")
feat_cols = feature_columns(df)
print(f"{len(df):,} games | {len(feat_cols)} features | events: {sorted(df['event'].unique())}")

9,752 games | 16 features | events: ['titled-tuesday-blitz-2026-02-10', 'titled-tuesday-blitz-2026-03-10', 'titled-tuesday-blitz-2026-04-14', 'titled-tuesday-blitz-2026-05-12', 'titled-tuesday-blitz-2026-06-09']


## 1. The data

Each row is one game. We keep only **rated, standard-chess, blitz** games (the
Titled Tuesday format) and label each from White's perspective. The class balance
is the first thing that shapes the problem:

In [2]:
counts = df["label"].value_counts()
pd.DataFrame({"count": counts, "share": (counts / len(df)).round(3)})

,count,share
label,,
win,4752,0.487
loss,4170,0.428
draw,830,0.085


Wins and losses are roughly balanced (~49% / ~43%); **draws are rare (~8.5%)**. That
imbalance is the central modeling challenge — a model can score ~49% accuracy by
never predicting a draw, so accuracy alone is misleading and we lean on log loss and
balanced accuracy too.

A peek at the feature matrix (identifier columns + features + label):

In [3]:
df.head(3)

,event,round_number,group_number,game_url,label,white_rating,black_rating,rating_diff,abs_rating_diff,mean_rating,elo_expected_white,in_tourn_white_points_pre,in_tourn_black_points_pre,in_tourn_points_diff_pre,in_tourn_white_streak,in_tourn_black_streak,in_tourn_white_games_played,in_tourn_black_games_played,in_tourn_white_recent_winrate,in_tourn_black_recent_winrate
0,titled-tuesday-blitz-2026-02-10,1,1,https://www.chess.com/game/live/165052787661,loss,2682,3316,-634,634,2999.0,0.025343,0.0,0.0,0.0,0,0,0,0,0.5,0.5
1,titled-tuesday-blitz-2026-02-10,1,1,https://www.chess.com/game/live/165052787663,win,3279,2678,601,601,2978.5,0.969517,0.0,0.0,0.0,0,0,0,0,0.5,0.5
2,titled-tuesday-blitz-2026-02-10,1,1,https://www.chess.com/game/live/165052787665,loss,2677,3252,-575,575,2964.5,0.035231,0.0,0.0,0.0,0,0,0,0,0.5,0.5


## 2. Features — and the leakage discipline

The API returns many fields per game, but **most describe how the game went**
(final result, PGN, accuracies, end-of-event standings) and are *not* admissible —
using them would leak the answer. The golden rule: a feature must be knowable
*before the first move*.

We enforce this structurally: features are produced only by registered extractors,
each declaring its output columns, so the admissible set is an explicit allow-list.

In [4]:
groups = {
    "ratings (at game time)": [c for c in feat_cols if "rating" in c],
    "elo expectation": [c for c in feat_cols if c.startswith("elo")],
    "round index": [c for c in feat_cols if c == "round_number"],
    "in-tournament form (rounds 1..r-1 only)": [c for c in feat_cols if c.startswith("in_tourn")],
}
for name, cols in groups.items():
    print(f"{name}:")
    for c in cols:
        print(f"    {c}")

ratings (at game time):
    white_rating
    black_rating
    rating_diff
    abs_rating_diff
    mean_rating
elo expectation:
    elo_expected_white
round index:
    round_number
in-tournament form (rounds 1..r-1 only):
    in_tourn_white_points_pre
    in_tourn_black_points_pre
    in_tourn_points_diff_pre
    in_tourn_white_streak
    in_tourn_black_streak
    in_tourn_white_games_played
    in_tourn_black_games_played
    in_tourn_white_recent_winrate
    in_tourn_black_recent_winrate


Two subtleties worth calling out in a walkthrough:

- **Ratings are the players' blitz ratings *at the time of the game*** (inside the
  game object), not their current rating — so they're genuinely pre-game.
- **In-tournament form is the one block that could leak.** It's reconstructed from
  a player's results in *earlier rounds of the same event*, so for round `r` we
  snapshot each player's state **before applying any round-`r` result**. Nothing
  from round `r` or later can influence round `r`'s features.

This leakage discipline isn't just convention — it's pinned down by tests. `uv run
pytest` (see `tests/test_leakage.py`) asserts that form never sees the current
round's result, that only the declared pre-game columns reach the model, and that
the train/test splits never overlap.

## 3. How we split the data, and why

Two complementary splits:

1. **Pooled stratified 75/25** — all five events pooled, random split (stratified on
   the label). Measures *in-distribution* performance.
2. **Temporal holdout** — train on the four earlier events (Feb–May), test on the
   latest (Jun 9). This mirrors real use: predicting *upcoming* games from *past*
   ones. It is the **honest** generalization number; the pooled split is slightly
   optimistic because games from the same tournament share context (same field,
   same day).

In [5]:
splits = [pooled_split(df), temporal_split(df)]
for s in splits:
    print(f"{s.name:55s} train={len(s.X_train):>5}  test={len(s.X_test):>5}")

pooled_75-25                                            train= 7314  test= 2438
temporal_holdout__test_titled-tuesday-blitz-2026-06-09  train= 7528  test= 2224


## 4. The models

A small, well-reasoned ladder — the brief prefers this to a black-box leaderboard
chaser. Each rung answers a specific question:

| model | question it answers |
|---|---|
| `majority` / `prior` | what do the naive floors look like? |
| `elo` | how far does the single Elo expectation get us? (the bar to beat) |
| `logistic` | do the other rating + form features add anything *linearly*? |
| `xgboost` | is there *non-linear* structure to exploit? |
| `logistic_balanced` | what does upweighting the rare draw class cost/buy? |

In [6]:
specs = default_model_specs(feat_cols)

rows = []
for split in splits:
    for spec in specs:
        m = fit_evaluate(spec, split)["metrics"]
        rows.append({
            "split": "pooled" if split.name.startswith("pooled") else "temporal",
            "model": spec.name,
            "accuracy": m["accuracy"],
            "balanced_acc": m["balanced_accuracy"],
            "macro_f1": m["macro_f1"],
            "recall_draw": m["recall_draw"],
            "log_loss": m["log_loss"],
        })
results = pd.DataFrame(rows)

In [7]:
for split_name, block in results.groupby("split", sort=False):
    print(f"\n=== {split_name} ===")
    show = block.drop(columns="split").sort_values("log_loss").reset_index(drop=True)
    print(show.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


=== pooled ===
            model  accuracy  balanced_acc  macro_f1  recall_draw  log_loss
          xgboost    0.7096        0.5173    0.4966       0.0048    0.7392
         logistic    0.6792        0.4950    0.4725       0.0000    0.7608
              elo    0.6801        0.4956    0.4731       0.0000    0.7689
            prior    0.4873        0.3333    0.2184       0.0000    0.9236
logistic_balanced    0.6325        0.5228    0.5176       0.2260    0.9335
         majority    0.4873        0.3333    0.2184       0.0000   17.7086

=== temporal ===
            model  accuracy  balanced_acc  macro_f1  recall_draw  log_loss
          xgboost    0.7032        0.5137    0.4900       0.0000    0.7545
         logistic    0.6749        0.4940    0.4705       0.0000    0.7699
              elo    0.6736        0.4930    0.4696       0.0000    0.7786
            prior    0.4852        0.3333    0.2178       0.0000    0.9307
logistic_balanced    0.6196        0.5048    0.5033       0.1910  

## 5. Reading the results

**Rating is most of the story.** Logistic regression on the single Elo expectation
already lands ~68% accuracy / ~0.77 log loss on the pooled split; adding the other
rating and form features barely moves it. That's expected — a player's live blitz
rating already integrates their recent form, and Elo is a purpose-built outcome
model.

**The non-linear model adds a small, *consistent* lift.** XGBoost is best on
**both** splits (log loss ~0.74–0.75 vs the linear models' ~0.77), so there is some
genuine non-linear structure (rating interactions, draw-prone regions). But the
margin over a one-feature Elo model is modest.

> *Aside worth showing in the walkthrough:* with only **two** events this same
> XGBoost **overfit** and was the *worst* model on the temporal split; with **five**
> it generalizes and comes out ahead. Model complexity has to scale with data — and
> the honest temporal split is what reveals the difference. It's why I evaluate on
> it rather than trusting the pooled number.

**Temporal ≈ pooled.** Every model loses only ~0.01–0.02 log loss going from the
pooled to the temporal split, so performance transfers across months — there's no
meaningful distribution shift between events.

**Draws are the hard part.** Let's look at what a calibrated model actually does:

In [8]:
# Confusion matrix for the interpretable logistic model on the honest temporal split.
logistic = next(s for s in specs if s.name == "logistic")
temporal = splits[1]
cm = fit_evaluate(logistic, temporal)["confusion"]
cm

,pred_win,pred_draw,pred_loss
true_win,804,0,275
true_draw,102,0,97
true_loss,249,0,697


The `draw` row is almost entirely misclassified as win/loss: at ~8.5% prevalence and
with ratings barely separating drawn games, the calibrated model essentially never
predicts a draw. `logistic_balanced` recovers meaningful draw recall (~20%) but pays
with lower accuracy and much worse calibration (log loss) — a genuine trade-off, not
a free win. Draws between titled players are close to *irreducibly* uncertain from
pre-game information alone.

## 6. Is it a good model?

**Yes, for what it is: a solid, well-calibrated model.** It clearly beats the naive
floors (log loss ~0.74 vs the prior's ~0.92) and improves modestly on a principled
Elo baseline. XGBoost is marginally best; given how small the gap is, shipping the
interpretable `logistic`/`elo` model is equally defensible, with XGBoost reserved for
when that last bit of log loss matters. It is good at what's knowable — rating
mismatches — and appropriately humble about what isn't: draws, and near-equal
pairings where the outcome is close to a coin flip.

It is **not** a model I'd expect to improve much without *new information*. The
ceiling here isn't model capacity — it's that pre-game rating already captures most
of the signal that exists.

## 7. What I'd do next

- **Player-history features (biggest expected lift):** mine each player's own game
  archives *before* the event — opening repertoire, decisive-vs-draw tendency,
  time-management/flagging style, recent form, head-to-head — with a strict
  `end_time < event_start` cutoff. This is the main source of *new* signal beyond
  rating. (It's a clean extension of the extractor pattern; I scoped it out here to
  respect the time box.)
- **Model the draw explicitly:** a two-stage decisive-vs-draw then win/loss model,
  or a draw-aware objective, instead of blunt class weights.
- **Calibration + light tuning:** isotonic/Platt calibration and a small
  hyperparameter search — judged on the *temporal* split so we don't reward
  overfitting.
- **More data:** the pipeline takes an arbitrary event list (five here); adding more
  would let the temporal split hold out several recent events instead of one,
  tightening the generalization estimate.
- **Testing:** the correctness-critical paths — label derivation and the leak-free
  feature/split construction — are already covered (`uv run pytest`, in `tests/`);
  I'd grow that suite alongside any new feature tiers.